<a href="https://colab.research.google.com/github/HasibulIslam007/git-hub-try/blob/main/caludeIsTheBest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile requirements.txt
torch>=2.0.0
torchvision>=0.15.0
numpy>=1.19.2
pyyaml>=5.4.1
tensorboard>=2.4.1

!pip install -r requirements.txt

Writing requirements.txt


In [2]:
!mkdir -p ssdl/configs
!mkdir -p ssdl/src/data
!mkdir -p ssdl/src/models
!mkdir -p ssdl/src/training
!mkdir -p ssdl/src/utils

In [3]:
%%writefile ssdl/configs/config.yaml
# Training settings
training:
  batch_size: 64  # reduced from 64
  epochs: 200
  learning_rate: 0.025
  momentum: 0.9
  weight_decay: 3.0e-4
  arch_learning_rate: 0.001
  arch_weight_decay: 1.0e-3
  grad_clip: 5.0
  lambda_latency: 0.1
  print_freq: 50

# Model settings
model:
  init_channels: 8
  layers: 4
  num_classes: 10
  drop_path_prob: 0.3
  steps: 3
  multiplier: 2
  hidden_dim: 256

# Search space settings
search_space:
  primitive_operations:
    - 'sep_conv_3x3'
    - 'sep_conv_5x5'
    - 'dil_conv_3x3'
    - 'dil_conv_5x5'
    - 'max_pool_3x3'
    - 'avg_pool_3x3'
    - 'skip_connect'

# Device settings
device:
  gpu_id: 0
  num_workers: 0

# Add these under the model section
adaptation:
  enabled: true  # Added this line
  error_threshold: 0.1
  adaptation_rate: 0.01
  momentum: 0.9
  window_size: 100

Writing ssdl/configs/config.yaml


In [4]:
%%writefile ssdl/src/utils/utils.py
import yaml
import torch
import random
import numpy as np
import os
from typing import Dict, Any

def load_config(config_path: str) -> Dict[str, Any]:
    """Load configuration from YAML file."""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def set_random_seed(seed: int = 42) -> None:
    """Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

def get_device(gpu_id: int = 0) -> torch.device:
    """Get device (CPU or GPU) for torch."""
    if torch.cuda.is_available():
        device = torch.device(f'cuda:{gpu_id}')
    else:
        device = torch.device('cpu')
    return device

class AverageMeter:
    """Computes and stores the average and current value."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

Writing ssdl/src/utils/utils.py


In [5]:
%%writefile ssdl/src/data/dataset.py
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import ssl
import os

def get_data_loaders(config):
    """Create train and validation data loaders"""

    # Handle SSL certificate verification
    ssl._create_default_https_context = ssl._create_unverified_context

    # Create data directory if it doesn't exist
    os.makedirs('./data', exist_ok=True)

    # Data augmentation and normalization for training
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                           (0.2023, 0.1994, 0.2010))
    ])

    # Just normalization for validation
    valid_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                           (0.2023, 0.1994, 0.2010))
    ])

    try:
        # Get datasets
        print("Downloading and preparing CIFAR-10 dataset...")
        train_data = datasets.CIFAR10(
            root='./data', train=True,
            download=True, transform=train_transform
        )

        valid_data = datasets.CIFAR10(
            root='./data', train=False,
            download=True, transform=valid_transform
        )

        # Create data loaders
        train_loader = DataLoader(
            train_data,
            batch_size=config['training']['batch_size'],
            shuffle=True,
            num_workers=config['device']['num_workers'],
            pin_memory=True
        )

        valid_loader = DataLoader(
            valid_data,
            batch_size=config['training']['batch_size'],
            shuffle=False,
            num_workers=config['device']['num_workers'],
            pin_memory=True
        )

        print("Dataset preparation completed successfully!")
        return train_loader, valid_loader

    except Exception as e:
        print(f"Error preparing dataset: {str(e)}")
        raise

Writing ssdl/src/data/dataset.py


In [6]:
%%writefile ssdl/src/models/operations.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class MixedOp(nn.Module):
    """Mixed operation that combines multiple operations"""
    def __init__(self, in_channels, out_channels, stride=1):
        super(MixedOp, self).__init__()

        # Define operations dictionary first
        self.ops_dict = {
            'conv_3x3': lambda: nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            ),
            'max_pool_3x3': lambda: nn.Sequential(
                nn.MaxPool2d(3, stride, padding=1),
                nn.Conv2d(in_channels, out_channels, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            ),
            'avg_pool_3x3': lambda: nn.Sequential(
                nn.AvgPool2d(3, stride, padding=1),
                nn.Conv2d(in_channels, out_channels, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            ),
            'skip_connect': lambda: nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            ) if stride != 1 or in_channels != out_channels else nn.Identity()
        }

        # Create operations
        self._ops = nn.ModuleList()
        for op_name in self.ops_dict:
            op = self.ops_dict[op_name]()
            self._ops.append(op)

        # Add architecture parameters
        self.arch_parameters = nn.Parameter(torch.randn(len(self.ops_dict)))
        self.arch_parameters.requires_grad = True

    def forward(self, x, weights=None):
        """
        Forward pass with architecture weights
        Args:
            x: input tensor
            weights: architecture weights for different operations
        """
        if weights is None:
            weights = F.softmax(self.arch_parameters, dim=-1)

        # Ensure weights is a tensor
        if not isinstance(weights, torch.Tensor):
            weights = torch.tensor(weights)

        # Ensure weights has the correct shape
        weights = weights.view(-1)

        return sum(w * op(x) for w, op in zip(weights, self._ops))

Writing ssdl/src/models/operations.py


In [7]:
%%writefile ssdl/src/models/ssdl.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SSDL(nn.Module):
    """Self-Supervised Decision Layer"""
    def __init__(self, in_channels, hidden_dim=256, num_classes=10):
        super(SSDL, self).__init__()

        self.in_channels = in_channels
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes

        # Feature encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 1),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True)
        )

        # Self-supervised head
        self.self_supervised_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Main classification head
        self.classification_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(hidden_dim, num_classes)
        )

        # Decision gate
        self.decision_gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x, return_features=False):
        # Extract features
        features = self.encoder(x)

        # Self-supervised representation
        self_sup_repr = self.self_supervised_head(features)

        # Classification logits
        logits = self.classification_head(features)

        # Decision gate output
        confidence = self.decision_gate(self_sup_repr)

        if return_features:
            return logits, confidence, self_sup_repr, features
        return logits, confidence

Writing ssdl/src/models/ssdl.py


In [8]:
%%writefile ssdl/src/models/search_space.py
import torch
import torch.nn as nn
from .operations import MixedOp
from .ssdl import SSDL  # Add this import

class Cell(nn.Module):
    # Your existing Cell class remains exactly the same
    def __init__(self, in_channels, out_channels, reduction=False):
        super(Cell, self).__init__()
        self.reduction = reduction
        stride = 2 if reduction else 1

        # Create mixed operations
        self._ops = nn.ModuleList([
            MixedOp(in_channels, out_channels, stride)
            for _ in range(2)  # Using 2 operations per cell for simplicity
        ])

    def forward(self, x, weights):
        # Ensure weights is a tensor and has correct shape
        if not isinstance(weights, torch.Tensor):
            weights = torch.tensor(weights)
        weights = weights.view(-1)

        # Apply operations with weights
        outputs = []
        for op, w in zip(self._ops, weights):
            outputs.append(op(x, w))
        return sum(outputs)

class SearchSpace(nn.Module):
    def __init__(self, config):
        super(SearchSpace, self).__init__()

        # Initial convolution
        self.stem = nn.Sequential(
            nn.Conv2d(3, config['init_channels'], 3, 1, 1, bias=False),
            nn.BatchNorm2d(config['init_channels'])
        )

        # Create cells
        self.cells = nn.ModuleList()
        in_channels = config['init_channels']

        for i in range(config['layers']):
            # Determine if current cell is reduction cell
            reduction = i in [config['layers']//3, 2*config['layers']//3]
            out_channels = in_channels * 2 if reduction else in_channels

            cell = Cell(in_channels, out_channels, reduction)
            self.cells.append(cell)

            in_channels = out_channels

        # Add SSDL layer (new)
        self.ssdl = SSDL(
            in_channels=in_channels,
            hidden_dim=config.get('hidden_dim', 256),
            num_classes=config['num_classes']
        )

        # Keep the original classifier for backward compatibility
        self.global_pooling = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(in_channels, config['num_classes'])

    def forward(self, x, weights_list, use_ssdl=True, return_features=False):
        x = self.stem(x)

        # Ensure weights_list is a list of tensors
        if not isinstance(weights_list, list):
            weights_list = [weights_list] * len(self.cells)

        # Pass through cells
        for cell, weights in zip(self.cells, weights_list):
            x = cell(x, weights)

        if use_ssdl:
            # Use SSDL for inference
            if return_features:
                logits, confidence, self_sup_repr, features = self.ssdl(x, return_features=True)
                return logits, confidence, self_sup_repr, features
            else:
                logits, confidence = self.ssdl(x)
                return logits, confidence
        else:
            # Use original classifier (for backward compatibility)
            x = self.global_pooling(x)
            x = x.view(x.size(0), -1)
            return self.classifier(x)

class MixedOp(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(MixedOp, self).__init__()

        # Define operations dictionary first
        self.ops_dict = {
            'conv_3x3': lambda: nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU()
            ),
            'max_pool_3x3': lambda: nn.Sequential(
                nn.MaxPool2d(3, stride, padding=1),
                nn.Conv2d(in_channels, out_channels, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            ),
            'avg_pool_3x3': lambda: nn.Sequential(
                nn.AvgPool2d(3, stride, padding=1),
                nn.Conv2d(in_channels, out_channels, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            ),
            'skip_connect': lambda: nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            ) if stride != 1 or in_channels != out_channels else nn.Identity()
        }

        # Create operations
        self._ops = nn.ModuleList()
        for op_name in self.ops_dict:
            op = self.ops_dict[op_name]()
            self._ops.append(op)

        # Add architecture parameters
        self.arch_parameters = nn.Parameter(torch.randn(len(self.ops_dict)))
        self.arch_parameters.requires_grad = True

    def forward(self, x, weights=None):
        """
        Forward pass with architecture weights
        Args:
            x: input tensor
            weights: architecture weights for different operations
        """
        if weights is None:
            weights = F.softmax(self.arch_parameters, dim=-1)

        # Ensure weights is a tensor
        if not isinstance(weights, torch.Tensor):
            weights = torch.tensor(weights)

        # Ensure weights has the correct shape
        weights = weights.view(-1)

        return sum(w * op(x) for w, op in zip(weights, self._ops))

Writing ssdl/src/models/search_space.py


In [9]:
%%writefile ssdl/src/models/adaptation.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from .search_space import SearchSpace

class ErrorDrivenAdapter(nn.Module):
    """Error-Driven Architecture Adaptation Mechanism"""
    def __init__(self, config):
        super(ErrorDrivenAdapter, self).__init__()

        self.error_threshold = config.get('error_threshold', 0.1)
        self.adaptation_rate = config.get('adaptation_rate', 0.01)
        self.momentum = config.get('momentum', 0.9)
        self.window_size = config.get('window_size', 100)

        # Error history for monitoring
        self.error_history = []
        self.moving_average = 0

        # Architecture parameters momentum
        self.arch_momentum = None

    def update_error_history(self, error):
        """Update error history and moving average"""
        self.error_history.append(error)
        if len(self.error_history) > self.window_size:
            self.error_history.pop(0)

        # Update moving average
        self.moving_average = sum(self.error_history) / len(self.error_history)

    def compute_adaptation_score(self, current_error):
        """Compute adaptation score based on error trends"""
        self.update_error_history(current_error)

        # Compute error trend
        error_trend = current_error - self.moving_average

        # Compute adaptation score
        adaptation_needed = error_trend > self.error_threshold
        adaptation_score = torch.sigmoid(torch.tensor(error_trend))

        return adaptation_needed, adaptation_score.item()

class AdaptiveSearchSpace(nn.Module):
    """SearchSpace with error-driven adaptation"""
    def __init__(self, config):
        super(AdaptiveSearchSpace, self).__init__()

        # Create base search space
        self.search_space = SearchSpace(config['model'])

        # Create error-driven adapter
        self.adapter = ErrorDrivenAdapter(config.get('adaptation', {}))

        # Error monitoring
        self.current_error = 0.0
        self.best_error = float('inf')

    def forward(self, x, weights_list, return_features=False):
        return self.search_space(x, weights_list, return_features=return_features)

    def adapt(self, error, confidence):
        """Perform architecture adaptation"""
        self.current_error = error

        # Update best error
        self.best_error = min(self.best_error, error)

        # Perform adaptation
        adapted, change = self.adapter.compute_adaptation_score(error)

        return adapted, change

Writing ssdl/src/models/adaptation.py


In [10]:
%%writefile ssdl/src/training/trainer.py
import torch
import torch.nn as nn
import torch.optim as optim
import time
import os
import numpy as np

class AverageMeter:
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

class Trainer:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)

        # Setup optimizers
        self.optimizer = optim.SGD(
            self.model.parameters(),
            lr=config['training']['learning_rate'],
            momentum=config['training']['momentum'],
            weight_decay=config['training']['weight_decay']
        )

        # Setup learning rate scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=config['training']['epochs']
        )

        # Setup loss functions
        self.criterion = nn.CrossEntropyLoss()

        # Setup metrics tracking
        self.best_acc = 0
        self.best_loss = float('inf')

    def train_epoch(self, train_loader, epoch):
        """Train for one epoch"""
        self.model.train()

        losses = AverageMeter()
        top1 = AverageMeter()
        batch_time = AverageMeter()

        end = time.time()
        for step, (inputs, targets) in enumerate(train_loader):
            inputs = inputs.to(self.device)
            targets = targets.to(self.device)

            # Generate weights for mixed operations
            weights_list = self._generate_arch_weights()

            # Forward pass
            logits, confidence = self.model(inputs, weights_list)
            loss = self.criterion(logits, targets)

            # Compute accuracy
            prec1 = self._accuracy(logits, targets)[0]
            n = inputs.size(0)
            losses.update(loss.item(), n)
            top1.update(prec1.item(), n)

            # Backward and optimize
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(),
                self.config['training']['grad_clip']
            )
            self.optimizer.step()

            # Measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if step % self.config['training'].get('print_freq', 50) == 0:
                print(f'Epoch: [{epoch}][{step}/{len(train_loader)}] '
                      f'Time {batch_time.val:.3f} ({batch_time.avg:.3f}) '
                      f'Loss {losses.val:.4f} ({losses.avg:.4f}) '
                      f'Prec@1 {top1.val:.3f} ({top1.avg:.3f})')

        return losses.avg, top1.avg

    def validate(self, valid_loader):
        """Validate the model"""
        self.model.eval()

        losses = AverageMeter()
        top1 = AverageMeter()

        with torch.no_grad():
            for step, (inputs, targets) in enumerate(valid_loader):
                inputs = inputs.to(self.device)
                targets = targets.to(self.device)

                # Generate weights for mixed operations
                weights_list = self._generate_arch_weights()

                # Forward pass
                logits, confidence = self.model(inputs, weights_list)
                loss = self.criterion(logits, targets)

                # Compute accuracy
                prec1 = self._accuracy(logits, targets)[0]
                n = inputs.size(0)
                losses.update(loss.item(), n)
                top1.update(prec1.item(), n)

                # Adapt architecture based on validation error
                try:
                    if self.config.get('adaptation', {}).get('enabled', False):
                        adapted, change = self.model.adapt(losses.avg, confidence)
                        if adapted:
                            print(f"Architecture adapted, change magnitude: {change:.6f}")
                except Exception as e:
                    print(f"Warning: Architecture adaptation failed: {str(e)}")

        print(f"\nValidation Results - Loss: {losses.avg:.4f} Acc: {top1.avg:.2f}%")
        return losses.avg, top1.avg

    def train(self, train_loader, valid_loader):
        """Main training loop"""
        try:
            for epoch in range(self.config['training']['epochs']):
                print(f"\nEpoch {epoch}")

                # Train for one epoch
                train_loss, train_acc = self.train_epoch(train_loader, epoch)

                # Validate
                valid_loss, valid_acc = self.validate(valid_loader)

                # Update learning rate
                self.scheduler.step()

                # Save best model
                if valid_acc > self.best_acc:
                    self.best_acc = valid_acc
                    self.save_checkpoint({
                        'epoch': epoch,
                        'state_dict': self.model.state_dict(),
                        'best_acc': self.best_acc,
                        'optimizer': self.optimizer.state_dict(),
                    }, is_best=True)

                print(f"\nEpoch Summary:")
                print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
                print(f"Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}%")
                print(f"Best Acc: {self.best_acc:.2f}%")

        except Exception as e:
            print(f"Error during training: {str(e)}")
            import traceback
            traceback.print_exc()
            raise

    def _generate_arch_weights(self):
        """Generate architecture weights for mixed operations"""
        weights_list = []
        for cell in self.model.search_space.cells:
            num_ops = len(cell._ops)
            weights = torch.ones(num_ops) / num_ops
            weights = weights.to(self.device)
            weights_list.append(weights)
        return weights_list

    def _accuracy(self, output, target, topk=(1,)):
        """Compute accuracy for top-k predictions"""
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

    def save_checkpoint(self, state, is_best):
        """Save checkpoint"""
        try:
            os.makedirs('checkpoints', exist_ok=True)
            filename = f"checkpoints/checkpoint_epoch{state['epoch']}.pth.tar"
            torch.save(state, filename)
            if is_best:
                best_filename = 'checkpoints/model_best.pth.tar'
                torch.save(state, best_filename)
                print(f"Saved best model checkpoint to {best_filename}")
        except Exception as e:
            print(f"Warning: Failed to save checkpoint: {str(e)}")

Writing ssdl/src/training/trainer.py


In [21]:
import os
import argparse
import torch
import sys

# Get the current directory in a way that works in Colab
current_dir = os.getcwd()
# Add the parent directory of ssdl to the Python path
sys.path.insert(0, os.path.abspath(os.path.join(current_dir, 'ssdl', '..')))


from ssdl.src.utils.utils import load_config, set_random_seed, get_device
from ssdl.src.models.operations import MixedOp
from ssdl.src.models.search_space import SearchSpace
from ssdl.src.models.adaptation import AdaptiveSearchSpace
from ssdl.src.training.trainer import Trainer
from ssdl.src.data.dataset import get_data_loaders

def parse_args():
    """Parse command line arguments"""
    parser = argparse.ArgumentParser(description='SSDL Training')
    parser.add_argument('--config', type=str, default='ssdl/configs/config.yaml', # Corrected path here
                      help='path to config file')
    parser.add_argument('--seed', type=int, default=42,
                      help='random seed')
    # Use parse_known_args to ignore arguments passed by the notebook environment
    args, unknown = parser.parse_known_args()
    return args

def test_mixed_op():
    """Test MixedOp separately"""
    print("\nStarting MixedOp test...")
    try:
        # Create input and output channels
        in_channels = 3
        out_channels = 8
        print(f"Channels: in={in_channels}, out={out_channels}")

        # Create MixedOp
        print("Creating MixedOp...")
        mixed_op = MixedOp(in_channels=in_channels, out_channels=out_channels)

        # Create dummy input
        print("Creating dummy input...")
        batch_size = 2
        dummy_input = torch.randn(batch_size, in_channels, 32, 32)
        print(f"Input shape: {dummy_input.shape}")

        # Create weights for each operation
        print("Creating weights...")
        num_ops = len(mixed_op._ops)
        weights = torch.ones(num_ops) / num_ops
        print(f"Number of operations: {num_ops}")
        print(f"Weights shape: {weights.shape}")

        # Forward pass
        print("Performing forward pass...")
        output = mixed_op(dummy_input, weights)
        print(f"Output shape: {output.shape}")

        print("MixedOp test completed successfully!")
        return True
    except Exception as e:
        print(f"Error in test_mixed_op: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

def test_search_space(config):
    """Test SearchSpace with dummy data"""
    print("\nStarting SearchSpace test...")
    try:
        # Create model
        print("Creating SearchSpace model...")
        model = SearchSpace(config['model'])

        # Create dummy input
        print("Creating dummy input...")
        batch_size = 2
        dummy_input = torch.randn(batch_size, 3, 32, 32)
        print(f"Input shape: {dummy_input.shape}")

        # Create dummy weights for each cell
        print("Creating weights for cells...")
        weights_list = []
        for cell in model.cells:
            num_ops = len(cell._ops)
            cell_weights = torch.ones(num_ops) / num_ops
            weights_list.append(cell_weights)
            print(f"Cell weights shape: {cell_weights.shape}")

        # Forward pass
        print("Performing forward pass...")
        logits, confidence = model(dummy_input, weights_list)
        print(f"Logits shape: {logits.shape}")
        print(f"Confidence shape: {confidence.shape}")

        # Test without SSDL (using original classifier)
        print("\nTesting original classifier...")
        output_original = model(dummy_input, weights_list, use_ssdl=False)
        print(f"Original classifier output shape: {output_original.shape}")

        print("SearchSpace test completed successfully!")
        return True
    except Exception as e:
        print(f"Error in test_search_space: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

def test_adaptation(config):
    """Test error-driven adaptation mechanism"""
    print("\nStarting Adaptation test...")
    try:
        # Create model
        print("Creating adaptive model...")
        adaptive_config = {
            'model': config['model'],
            'adaptation': config.get('adaptation', {
                'error_threshold': 0.1,
                'adaptation_rate': 0.01,
                'momentum': 0.9,
                'window_size': 100
            })
        }
        model = AdaptiveSearchSpace(adaptive_config)

        # Create dummy input
        print("Creating dummy input...")
        batch_size = 2
        dummy_input = torch.randn(batch_size, 3, 32, 32)
        print(f"Input shape: {dummy_input.shape}")

        # Create dummy weights
        print("Creating weights...")
        weights_list = []
        for cell in model.search_space.cells:
            num_ops = len(cell._ops)
            cell_weights = torch.ones(num_ops) / num_ops
            weights_list.append(cell_weights)

        # Forward pass
        print("Performing forward pass...")
        logits, confidence = model(dummy_input, weights_list)
        print(f"Logits shape: {logits.shape}")
        print(f"Confidence shape: {confidence.shape}")

        # Test adaptation
        print("\nTesting adaptation mechanism...")
        dummy_error = 0.15  # Error above threshold
        adapted, change = model.adapt(dummy_error, confidence)
        print(f"Adaptation performed: {adapted}")
        print(f"Architecture change magnitude: {change:.6f}")

        # Test adaptation with low error
        print("\nTesting adaptation with low error...")
        dummy_error = 0.05  # Error below threshold
        adapted, change = model.adapt(dummy_error, confidence)
        print(f"Adaptation performed: {adapted}")
        print(f"Architecture change magnitude: {change:.6f}")

        print("\nAdaptation test completed successfully!")
        return True
    except Exception as e:
        print(f"Error in test_adaptation: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

def train_model(config):
    """Train the model"""
    print("\nStarting model training...")
    try:
        # Create model
        model = AdaptiveSearchSpace(config)

        # Create trainer
        trainer = Trainer(model, config)

        # Get data loaders
        train_loader, valid_loader = get_data_loaders(config)

        # Train the model
        trainer.train(train_loader, valid_loader)

        print("Training completed successfully!")
        return True
    except Exception as e:
        print(f"Error in training: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

def main():
    try:
        print("Starting main function...")

        # Parse arguments
        print("Parsing arguments...")
        args = parse_args()

        # Load configuration
        print("Loading configuration...")
        config = load_config(args.config)

        # Set random seed
        print("Setting random seed...")
        set_random_seed(args.seed)

        # Get device
        print("Getting device...")
        device = get_device(config['device']['gpu_id'])
        print(f"Using device: {device}")

        # Run tests
        mixed_op_success = test_mixed_op()

        search_space_success = False
        if mixed_op_success:
            search_space_success = test_search_space(config)

        adaptation_success = False
        if search_space_success:
            adaptation_success = test_adaptation(config)

        training_success = False
        if adaptation_success:
            training_success = train_model(config)

        # Final status
        if mixed_op_success and search_space_success and adaptation_success and training_success:
            print("\nAll operations completed successfully!")
        else:
            print("\nSome operations failed!")

    except Exception as e:
        print(f"Error in main: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == '__main__':
    print("Script starting...")
    main()
    print("Script finished.")

Script starting...
Starting main function...
Parsing arguments...
Loading configuration...
Setting random seed...
Getting device...
Using device: cpu

Starting MixedOp test...
Channels: in=3, out=8
Creating MixedOp...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights...
Number of operations: 4
Weights shape: torch.Size([4])
Performing forward pass...
Output shape: torch.Size([2, 8, 32, 32])
MixedOp test completed successfully!

Starting SearchSpace test...
Creating SearchSpace model...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights for cells...
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Performing forward pass...
Logits shape: torch.Size([2, 10])
Confidence shape: torch.Size([2, 1])

Testing original classifier...
Original classifier output shape: torch.Size([2, 10])
SearchSpace test completed successfully!

Starting Adapt

Traceback (most recent call last):
  File "/content/ssdl/src/training/trainer.py", line 202, in train
    train_loss, train_acc = self.train_epoch(train_loader, epoch)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ssdl/src/training/trainer.py", line 114, in train_epoch
    [p for p in self.model.parameters() if p.requires_grad and 'adapter.arch_parameters' not in name],
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/ssdl/src/training/trainer.py", line 114, in <listcomp>
    [p for p in self.model.parameters() if p.requires_grad and 'adapter.arch_parameters' not in name],
                                                                                                ^^^^
NameError: name 'name' is not defined
Traceback (most recent call last):
  File "<ipython-input-21-2313067029>", line 184, in train_model
    trainer.train(train_loader, valid_loader)
  File "/content/ssdl/src/tr

In [12]:
%run ssdl/main.py

Script starting...
Starting main function...
Parsing arguments...
Loading configuration...
Setting random seed...
Getting device...
Using device: cpu

Starting MixedOp test...
Channels: in=3, out=8
Creating MixedOp...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights...
Number of operations: 4
Weights shape: torch.Size([4])
Performing forward pass...
Output shape: torch.Size([2, 8, 32, 32])
MixedOp test completed successfully!

Starting SearchSpace test...
Creating SearchSpace model...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights for cells...
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Performing forward pass...
Logits shape: torch.Size([2, 10])
Confidence shape: torch.Size([2, 1])

Testing original classifier...
Original classifier output shape: torch.Size([2, 10])
SearchSpace test completed successfully!

Starting Adapt

100%|██████████| 170M/170M [00:02<00:00, 79.4MB/s]


Dataset preparation completed successfully!

Epoch 0
Epoch: [0][0/782] Time 0.323 (0.323) Loss 2.3055 (2.3055) Prec@1 10.938 (10.938)
Epoch: [0][50/782] Time 0.163 (0.182) Loss 1.8098 (2.0768) Prec@1 34.375 (20.312)
Epoch: [0][100/782] Time 0.193 (0.190) Loss 1.8745 (1.9872) Prec@1 26.562 (23.747)
Epoch: [0][150/782] Time 0.183 (0.195) Loss 1.8792 (1.9208) Prec@1 17.188 (26.625)
Epoch: [0][200/782] Time 0.251 (0.193) Loss 1.9078 (1.8700) Prec@1 17.188 (28.591)
Epoch: [0][250/782] Time 0.188 (0.195) Loss 1.4203 (1.8233) Prec@1 31.250 (30.603)
Epoch: [0][300/782] Time 0.179 (0.199) Loss 1.6897 (1.7870) Prec@1 37.500 (31.920)
Epoch: [0][350/782] Time 0.170 (0.200) Loss 1.4212 (1.7537) Prec@1 43.750 (33.400)
Epoch: [0][400/782] Time 0.232 (0.199) Loss 1.2548 (1.7238) Prec@1 56.250 (34.757)
Epoch: [0][450/782] Time 0.182 (0.198) Loss 1.5208 (1.6956) Prec@1 50.000 (35.931)
Epoch: [0][500/782] Time 0.203 (0.199) Loss 1.5475 (1.6718) Prec@1 39.062 (36.926)
Epoch: [0][550/782] Time 0.170 (0.199

KeyboardInterrupt: 

# Task
Enhance the adaptive search space model by implementing architecture adaptation logic based on the adaptation score, integrating this adaptation into the training process, and exploring different adaptation strategies.

## Implement architecture adaptation logic

### Subtask:
Modify the `adapt` method in `ssdl/src/models/adaptation.py` to translate the adaptation score into concrete changes in the architecture parameters within the `SearchSpace` model. This could involve updating the `arch_parameters` in the `MixedOp` layers based on the computed adaptation score and potentially the confidence from the SSDL.


**Reasoning**:
Modify the `adapt` method in `ssdl/src/models/adaptation.py` to update the architecture parameters based on the adaptation score and confidence.



In [13]:
%%writefile ssdl/src/models/adaptation.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from .search_space import SearchSpace

class ErrorDrivenAdapter(nn.Module):
    """Error-Driven Architecture Adaptation Mechanism"""
    def __init__(self, config):
        super(ErrorDrivenAdapter, self).__init__()

        self.error_threshold = config.get('error_threshold', 0.1)
        self.adaptation_rate = config.get('adaptation_rate', 0.01)
        self.momentum = config.get('momentum', 0.9)
        self.window_size = config.get('window_size', 100)

        # Error history for monitoring
        self.error_history = []
        self.moving_average = 0

        # Architecture parameters momentum
        self.arch_momentum = None

    def update_error_history(self, error):
        """Update error history and moving average"""
        self.error_history.append(error)
        if len(self.error_history) > self.window_size:
            self.error_history.pop(0)

        # Update moving average
        self.moving_average = sum(self.error_history) / len(self.error_history)

    def compute_adaptation_score(self, current_error):
        """Compute adaptation score based on error trends"""
        self.update_error_history(current_error)

        # Compute error trend
        error_trend = current_error - self.moving_average

        # Compute adaptation score
        adaptation_needed = error_trend > self.error_threshold
        adaptation_score = torch.sigmoid(torch.tensor(error_trend))

        return adaptation_needed, adaptation_score.item()

class AdaptiveSearchSpace(nn.Module):
    """SearchSpace with error-driven adaptation"""
    def __init__(self, config):
        super(AdaptiveSearchSpace, self).__init__()

        # Create base search space
        self.search_space = SearchSpace(config['model'])

        # Create error-driven adapter
        self.adapter = ErrorDrivenAdapter(config.get('adaptation', {}))

        # Error monitoring
        self.current_error = 0.0
        self.best_error = float('inf')

    def forward(self, x, weights_list, return_features=False):
        return self.search_space(x, weights_list, return_features=return_features)

    def adapt(self, error, confidence):
        """Perform architecture adaptation"""
        self.current_error = error

        # Update best error
        self.best_error = min(self.best_error, error)

        # Perform adaptation
        adapted, change = self.adapter.compute_adaptation_score(error)

        if adapted:
            print(f"Adapting architecture with change magnitude: {change:.6f}")
            # Translate adaptation score into changes in architecture parameters
            # Iterate through cells and their mixed operations
            for cell in self.search_space.cells:
                for op in cell._ops:
                    # Simple adaptation strategy: add a small value proportional to change
                    # Also incorporate confidence: adapt more when confidence is low
                    # A higher error trend (leading to higher change) and lower confidence
                    # should lead to a larger adaptation.
                    # Example: update = change * (1 - confidence.mean()) * self.adapter.adaptation_rate
                    # Ensure confidence is a scalar or has a meaningful mean
                    if isinstance(confidence, torch.Tensor):
                         # Use mean if confidence is a tensor with multiple values
                        confidence_scalar = confidence.mean().item()
                    else:
                        # Assume confidence is already a scalar
                        confidence_scalar = confidence

                    update = change * (1.0 - confidence_scalar) * self.adapter.adaptation_rate

                    # Apply update to architecture parameters
                    with torch.no_grad():
                        op.arch_parameters.add_(update) # Simple additive update

        return adapted, change

Overwriting ssdl/src/models/adaptation.py


## Update trainer to utilize adaptation

### Subtask:
Modify the `Trainer` class in `ssdl/src/training/trainer.py` to call the enhanced `adapt` method of the `AdaptiveSearchSpace` model during validation and apply the resulting architectural changes before the next training epoch.


**Reasoning**:
I need to modify the `Trainer` class in `ssdl/src/training/trainer.py` to ensure the `self.model.adapt` method is called correctly within the `validate` method with the validation loss and confidence.



In [14]:
%%writefile ssdl/src/training/trainer.py
import torch
import torch.nn as nn
import torch.optim as optim
import time
import os
import numpy as np

class AverageMeter:
    """Computes and stores the average and current value."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

class Trainer:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)

        # Setup optimizers
        self.optimizer = optim.SGD(
            self.model.parameters(),
            lr=config['training']['learning_rate'],
            momentum=config['training']['momentum'],
            weight_decay=config['training']['weight_decay']
        )

        # Setup learning rate scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=config['training']['epochs']
        )

        # Setup loss functions
        self.criterion = nn.CrossEntropyLoss()

        # Setup metrics tracking
        self.best_acc = 0
        self.best_loss = float('inf')

    def train_epoch(self, train_loader, epoch):
        """Train for one epoch"""
        self.model.train()

        losses = AverageMeter()
        top1 = AverageMeter()
        batch_time = AverageMeter()

        end = time.time()
        for step, (inputs, targets) in enumerate(train_loader):
            inputs = inputs.to(self.device)
            targets = targets.to(self.device)

            # Generate weights for mixed operations
            weights_list = self._generate_arch_weights()

            # Forward pass
            logits, confidence = self.model(inputs, weights_list)
            loss = self.criterion(logits, targets)

            # Compute accuracy
            prec1 = self._accuracy(logits, targets)[0]
            n = inputs.size(0)
            losses.update(loss.item(), n)
            top1.update(prec1.item(), n)

            # Backward and optimize
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                self.model.parameters(),
                self.config['training']['grad_clip']
            )
            self.optimizer.step()

            # Measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if step % self.config['training'].get('print_freq', 50) == 0:
                print(f'Epoch: [{epoch}][{step}/{len(train_loader)}] '
                      f'Time {batch_time.val:.3f} ({batch_time.avg:.3f}) '
                      f'Loss {losses.val:.4f} ({losses.avg:.4f}) '
                      f'Prec@1 {top1.val:.3f} ({top1.avg:.3f})')

        return losses.avg, top1.avg

    def validate(self, valid_loader):
        """Validate the model"""
        self.model.eval()

        losses = AverageMeter()
        top1 = AverageMeter()

        # Store confidence values from the validation batches
        all_confidences = []

        with torch.no_grad():
            for step, (inputs, targets) in enumerate(valid_loader):
                inputs = inputs.to(self.device)
                targets = targets.to(self.device)

                # Generate weights for mixed operations
                weights_list = self._generate_arch_weights()

                # Forward pass
                logits, confidence = self.model(inputs, weights_list)
                loss = self.criterion(logits, targets)

                # Compute accuracy
                prec1 = self._accuracy(logits, targets)[0]
                n = inputs.size(0)
                losses.update(loss.item(), n)
                top1.update(prec1.item(), n)

                # Store confidence values
                all_confidences.append(confidence.cpu().numpy())

        print(f"\nValidation Results - Loss: {losses.avg:.4f} Acc: {top1.avg:.2f}%")

        # Compute average confidence over the validation set
        avg_confidence = np.mean(all_confidences) if all_confidences else 0.0

        # Adapt architecture based on validation error and average confidence
        try:
            if self.config.get('adaptation', {}).get('enabled', False):
                adapted, change = self.model.adapt(losses.avg, avg_confidence) # Pass average confidence
                if adapted:
                    print(f"Architecture adapted, change magnitude: {change:.6f}")
        except Exception as e:
            print(f"Warning: Architecture adaptation failed: {str(e)}")


        return losses.avg, top1.avg

    def train(self, train_loader, valid_loader):
        """Main training loop"""
        try:
            for epoch in range(self.config['training']['epochs']):
                print(f"\nEpoch {epoch}")

                # Train for one epoch
                train_loss, train_acc = self.train_epoch(train_loader, epoch)

                # Validate
                valid_loss, valid_acc = self.validate(valid_loader)

                # Update learning rate
                self.scheduler.step()

                # Save best model
                if valid_acc > self.best_acc:
                    self.best_acc = valid_acc
                    self.save_checkpoint({
                        'epoch': epoch,
                        'state_dict': self.model.state_dict(),
                        'best_acc': self.best_acc,
                        'optimizer': self.optimizer.state_dict(),
                    }, is_best=True)

                print(f"\nEpoch Summary:")
                print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
                print(f"Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}%")
                print(f"Best Acc: {self.best_acc:.2f}%")

        except Exception as e:
            print(f"Error during training: {str(e)}")
            import traceback
            traceback.print_exc()
            raise

    def _generate_arch_weights(self):
        """Generate architecture weights for mixed operations"""
        weights_list = []
        for cell in self.model.search_space.cells:
            num_ops = len(cell._ops)
            # Use softmax of the learned architecture parameters
            weights = F.softmax(torch.stack([op.arch_parameters for op in cell._ops]), dim=-1)
            weights = weights.mean(dim=0) # Average weights across ops in a cell
            weights_list.append(weights)
        return weights_list


    def _accuracy(self, output, target, topk=(1,)):
        """Compute accuracy for top-k predictions"""
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

    def save_checkpoint(self, state, is_best):
        """Save checkpoint"""
        try:
            os.makedirs('checkpoints', exist_ok=True)
            filename = f"checkpoints/checkpoint_epoch{state['epoch']}.pth.tar"
            torch.save(state, filename)
            if is_best:
                best_filename = 'checkpoints/model_best.pth.tar'
                torch.save(state, best_filename)
                print(f"Saved best model checkpoint to {best_filename}")
        except Exception as e:
            print(f"Warning: Failed to save checkpoint: {str(e)}")

Overwriting ssdl/src/training/trainer.py


## Refine adaptation strategy

### Subtask:
Explore different strategies for applying the adaptation, such as:

*   Gradually updating architecture parameters based on the adaptation score.
*   Using the adaptation score to influence a selection process of operations in the `MixedOp`.
*   Considering the confidence score from the SSDL in the adaptation decision.


**Reasoning**:
Modify the adaptation logic in `ssdl/src/models/adaptation.py` to implement different strategies for applying adaptation, including using a weighted average update, influencing softmax temperature, and refining confidence incorporation.



In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from .search_space import SearchSpace
import numpy as np # Import numpy
import logging # Import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


class ErrorDrivenAdapter(nn.Module):
    """Error-Driven Architecture Adaptation Mechanism"""
    def __init__(self, config):
        super(ErrorDrivenAdapter, self).__init__()

        self.error_threshold = config.get('error_threshold', 0.1)
        self.adaptation_rate = config.get('adaptation_rate', 0.01)
        self.momentum = config.get('momentum', 0.9)
        self.window_size = config.get('window_size', 100)
        self.temperature_decay_rate = config.get('temperature_decay_rate', 0.9) # Added temperature decay
        self.min_temperature = config.get('min_temperature', 0.1) # Added minimum temperature

        # Error history for monitoring
        self.error_history = []
        self.moving_average = 0

        # Architecture parameters momentum (initialized in AdaptiveSearchSpace)
        self.arch_momentum = None

        # Temperature for softmax (initialized in AdaptiveSearchSpace)
        self.temperature = 1.0 # Initial temperature

    def update_error_history(self, error):
        """Update error history and moving average"""
        self.error_history.append(error)
        if len(self.error_history) > self.window_size:
            self.error_history.pop(0)

        # Update moving average
        self.moving_average = sum(self.error_history) / len(self.error_history)

    def compute_adaptation_score(self, current_error):
        """Compute adaptation score based on error trends"""
        self.update_error_history(current_error)

        # Compute error trend
        error_trend = current_error - self.moving_average

        # Compute adaptation needed flag
        adaptation_needed = error_trend > self.error_threshold

        # Compute adaptation score (sigmoid of error trend)
        adaptation_score = torch.sigmoid(torch.tensor(error_trend))

        return adaptation_needed, adaptation_score.item()

    def adapt_architecture(self, search_space, adaptation_score, confidence):
        """Adapt architecture parameters based on score and confidence"""
        logging.info("--- Adapting Architecture ---")
        with torch.no_grad():
            for i, cell in enumerate(search_space.cells):
                logging.info(f"  Adapting Cell {i}")
                for j, op in enumerate(cell._ops):
                    # Log parameters before adaptation
                    logging.info(f"    MixedOp {j} Arch Params Before: {op.arch_parameters.cpu().numpy()}")

                    # Strategy 1: Weighted average update
                    # Combine current parameters with a direction influenced by error and confidence
                    # A higher adaptation score and lower confidence suggest a need to change.
                    # Direction could be proportional to the gradient of error wrt arch params (if computable)
                    # or a random direction scaled by the adaptation score and confidence.
                    # For simplicity, let's use a random direction scaled by adaptation and inversely by confidence.

                    # Ensure confidence is a scalar
                    if isinstance(confidence, torch.Tensor):
                        confidence_scalar = confidence.mean().item()
                    else:
                        confidence_scalar = confidence

                    # Scale adaptation based on confidence (more adaptation when confidence is low)
                    # Add a small epsilon to avoid division by zero if confidence is exactly 0.
                    scaled_adaptation_rate = self.adaptation_rate * adaptation_score * (1.0 / (confidence_scalar + 1e-6))

                    # Generate a random direction (could also be based on gradients if available)
                    direction = torch.randn_like(op.arch_parameters)
                    direction = direction / (direction.norm() + 1e-6) # Normalize direction

                    # Update architecture parameters using weighted average
                    # new_params = (1 - alpha) * current_params + alpha * (current_params + scaled_direction)
                    # where alpha is the learning rate / adaptation rate
                    # or simply: current_params = current_params + scaled_direction
                    op.arch_parameters.add_(direction * scaled_adaptation_rate)

                    # Log parameters after adaptation
                    logging.info(f"    MixedOp {j} Arch Params After: {op.arch_parameters.cpu().numpy()}")


                    # Optional: Apply momentum to architecture parameters (requires storing momentum state)
                    # This part is commented out as it requires more complex state management
                    # if self.arch_momentum is None or self.arch_momentum.shape != op.arch_parameters.shape:
                    #     self.arch_momentum = torch.zeros_like(op.arch_parameters)
                    # self.arch_momentum = self.momentum * self.arch_momentum + (1 - self.momentum) * (direction * scaled_adaptation_rate)
                    # op.arch_parameters.copy_(op.arch_parameters + self.arch_momentum)

    def adjust_softmax_temperature(self, adaptation_score):
        """Adjust the temperature for softmax over architecture parameters"""
        # Strategy 2: Influence softmax temperature
        # Higher adaptation score (higher error) -> increase temperature (more exploration)
        # Lower adaptation score (lower error) -> decrease temperature (more exploitation)
        # We can use a simple decay towards a minimum temperature, with an increase based on adaptation score.
        decayed_temp = self.temperature * self.temperature_decay_rate
        increased_temp = 1.0 + adaptation_score # Increase temperature based on adaptation score (0 to 1)

        # Combine decay and increase, ensuring it stays above min_temperature
        self.temperature = max(self.min_temperature, decayed_temp * increased_temp)

        # Ensure temperature is within a reasonable range (e.g., not excessively high)
        self.temperature = min(self.temperature, 5.0) # Cap temperature at a reasonable value
        logging.info(f"Softmax temperature adjusted to: {self.temperature:.4f}")


class AdaptiveSearchSpace(nn.Module):
    """SearchSpace with error-driven adaptation"""
    def __init__(self, config):
        super(AdaptiveSearchSpace, self).__init__()

        # Create base search space
        self.search_space = SearchSpace(config['model'])

        # Create error-driven adapter
        self.adapter = ErrorDrivenAdapter(config.get('adaptation', {}))

        # Error monitoring
        self.current_error = 0.0
        self.best_error = float('inf')

    def forward(self, x, weights_list=None, use_ssdl=True, return_features=False):
         # If weights_list is not provided, generate them using the adapter's temperature
        if weights_list is None:
             weights_list = self._generate_arch_weights()

        return self.search_space(x, weights_list, use_ssdl=use_ssdl, return_features=return_features)

    def _generate_arch_weights(self):
        """Generate architecture weights for mixed operations using the adapter's temperature"""
        weights_list = []
        for cell in self.search_space.cells:
            # Corrected weight generation based on Cell structure
            cell_weights = []
            for op in cell._ops:
                 # Apply softmax with temperature to each MixedOp's parameters
                 op_weights = F.softmax(op.arch_parameters / self.adapter.temperature, dim=-1)
                 cell_weights.append(op_weights)
            # Flatten the list of weights for each MixedOp within the cell into a single tensor for the cell
            # This assumes Cell's forward expects a single tensor of weights for all its internal MixedOps
            # which will then be used for the internal operations of each MixedOp.
            # Re-reading SearchSpace and Cell:
            # SearchSpace passes weights_list (list of tensors, one per Cell) to Cell.
            # Cell's forward takes a single `weights` tensor.
            # Cell's forward then iterates through its `_ops` (list of MixedOps) and calls op(x, w).
            # This means the `weights` tensor passed to Cell is used for the internal operations of *each* MixedOp in that cell.
            # So, `_generate_arch_weights` should return a list of tensors, where each tensor is the architecture weights for the internal operations of a MixedOp.
            # And the SearchSpace forward should iterate through its cells and *their* MixedOps, passing the corresponding weight tensor from weights_list.

            # Let's adjust _generate_arch_weights to return a list of weights for *each MixedOp* in the search space.
            for op in cell._ops: # Iterate through the MixedOps within the cell
                 # Apply softmax with temperature to each MixedOp's parameters
                 op_weights = F.softmax(op.arch_parameters / self.adapter.temperature, dim=-1)
                 weights_list.append(op_weights)

        return weights_list # This will be a list of tensors, one for each MixedOp in the entire network


    def adapt(self, error, confidence):
        """Perform architecture adaptation"""
        self.current_error = error

        # Update best error
        self.best_error = min(self.best_error, error)

        # Perform adaptation
        adapted, adaptation_score = self.adapter.compute_adaptation_score(error)

        if adapted:
            logging.info(f"Architecture adapted with adaptation score: {adaptation_score:.6f}")
            # Apply architecture adaptation strategy
            self.adapter.adapt_architecture(self.search_space, adaptation_score, confidence)

        # Always adjust temperature based on adaptation score (even if not 'adapted' flag is False)
        # The score reflects the error trend, which is useful for temperature adjustment regardless of threshold.
        if hasattr(self, 'adapter'): # Check if adapter exists
             self.adapter.adjust_softmax_temperature(adaptation_score)


        return adapted, adaptation_score # Return adaptation_score for clarity

ImportError: attempted relative import with no known parent package

**Reasoning**:
The adaptation logic has been modified in `ssdl/src/models/adaptation.py`. Now update the `Trainer` class to correctly use the potentially adjusted architecture weights generated by the `AdaptiveSearchSpace` which now considers the temperature.



In [22]:
%%writefile ssdl/src/training/trainer.py
import torch
import torch.nn as nn
import torch.optim as optim
import time
import os
import numpy as np
import torch.nn.functional as F # Ensure F is imported for softmax

class AverageMeter:
    """Computes and stores the average and current value."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

class Trainer:
    def __init__(self, model, config):
        self.model = model
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)

        # Setup optimizers
        # The optimizer should only optimize the model's network parameters,
        # not the architecture parameters, as those are updated by the adapter.
        # We need to identify the network parameters to optimize.
        # Assuming model.search_space contains the network parameters
        # and adapter has the arch_parameters which are not in model.parameters()
        # Or perhaps the adapter's parameters are part of the model's parameters,
        # but we exclude them from this optimizer.

        # Let's find parameters that are NOT the adapter's architecture parameters
        network_params = []
        arch_param_names = set()
        for name, param in self.model.named_parameters():
            if 'adapter.arch_parameters' in name:
                arch_param_names.add(name)
            else:
                network_params.append(param)

        if not network_params:
             # Fallback if no network parameters are found (shouldn't happen with this model)
             print("Warning: No network parameters found for optimization!")
             network_params = list(self.model.parameters())


        self.optimizer = optim.SGD(
            network_params, # Optimize only network parameters
            lr=config['training']['learning_rate'],
            momentum=config['training']['momentum'],
            weight_decay=config['training']['weight_decay']
        )

        # Setup learning rate scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=config['training']['epochs']
        )

        # Setup loss functions
        self.criterion = nn.CrossEntropyLoss()

        # Setup metrics tracking
        self.best_acc = 0
        self.best_loss = float('inf')

    def train_epoch(self, train_loader, epoch):
        """Train for one epoch"""
        self.model.train()

        losses = AverageMeter()
        top1 = AverageMeter()
        batch_time = AverageMeter()

        end = time.time()
        for step, (inputs, targets) in enumerate(train_loader):
            inputs = inputs.to(self.device)
            targets = targets.to(self.device)

            # Get architecture weights from the model (uses the adapter's temperature)
            # The model's forward pass will now handle weight generation if not provided
            # logits, confidence = self.model(inputs) # Model generates weights internally

            # However, the current SearchSpace forward expects weights_list.
            # The AdaptiveSearchSpace forward pass also takes weights_list.
            # Let's update Trainer to explicitly get weights from the model (which uses the adapter).
            weights_list = self.model._generate_arch_weights() # Get weights using adapter's temperature

            # Forward pass using generated weights
            logits, confidence = self.model(inputs, weights_list=weights_list)
            loss = self.criterion(logits, targets)

            # Compute accuracy
            prec1 = self._accuracy(logits, targets)[0]
            n = inputs.size(0)
            losses.update(loss.item(), n)
            top1.update(prec1.item(), n)

            # Backward and optimize network parameters
            self.optimizer.zero_grad()
            loss.backward()
            # Clip gradients for network parameters
            # Corrected: Iterate over named parameters to check the name
            torch.nn.utils.clip_grad_norm_(
                [param for name, param in self.model.named_parameters() if param.requires_grad and 'adapter.arch_parameters' not in name],
                self.config['training']['grad_clip']
            )
            self.optimizer.step()

            # Measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if step % self.config['training'].get('print_freq', 50) == 0:
                print(f'Epoch: [{epoch}][{step}/{len(train_loader)}] '
                      f'Time {batch_time.val:.3f} ({batch_time.avg:.3f}) '
                      f'Loss {losses.val:.4f} ({losses.avg:.4f}) '
                      f'Prec@1 {top1.val:.3f} ({top1.avg:.3f})')

        return losses.avg, top1.avg

    def validate(self, valid_loader):
        """Validate the model"""
        self.model.eval()

        losses = AverageMeter()
        top1 = AverageMeter()

        # Store confidence values from the validation batches
        all_confidences = []

        with torch.no_grad():
            for step, (inputs, targets) in enumerate(valid_loader):
                inputs = inputs.to(self.device)
                targets = targets.to(self.device)

                # Get architecture weights from the model (uses the adapter's temperature)
                weights_list = self.model._generate_arch_weights()

                # Forward pass using generated weights
                logits, confidence = self.model(inputs, weights_list=weights_list)
                loss = self.criterion(logits, targets)

                # Compute accuracy
                prec1 = self._accuracy(logits, targets)[0]
                n = inputs.size(0)
                losses.update(loss.item(), n)
                top1.update(prec1.item(), n)

                # Store confidence values
                # Detach confidence from graph and move to CPU before storing
                all_confidences.append(confidence.detach().cpu().numpy())


        print(f"\nValidation Results - Loss: {losses.avg:.4f} Acc: {top1.avg:.2f}%")

        # Compute average confidence over the validation set
        avg_confidence = np.mean(all_confidences) if all_confidences else 0.0

        # Adapt architecture based on validation error and average confidence
        try:
            # Check if adaptation is enabled
            adaptation_enabled = self.config.get('adaptation', {}).get('enabled', False)
            if adaptation_enabled:
                # Call the model's adapt method, which now handles adaptation logic internally
                adapted, change = self.model.adapt(losses.avg, avg_confidence)
                if adapted:
                    print(f"Architecture adapted, change magnitude: {change:.6f}")
            else:
                 # If adaptation is not enabled, still adjust temperature based on validation loss
                 # The adapter's compute_adaptation_score and adjust_softmax_temperature
                 # are useful even without architectural parameter updates based on threshold.
                 # We can call these methods directly on the adapter instance within the model.
                 # Calculate score without triggering full adaptation logic if not enabled.
                 if hasattr(self.model, 'adapter'):
                     _, adaptation_score = self.model.adapter.compute_adaptation_score(losses.avg)
                     self.model.adapter.adjust_softmax_temperature(adaptation_score)


        except Exception as e:
            print(f"Warning: Architecture adaptation failed: {str(e)}")


        return losses.avg, top1.avg

    def train(self, train_loader, valid_loader):
        """Main training loop"""
        try:
            for epoch in range(self.config['training']['epochs']):
                print(f"\nEpoch {epoch}")

                # Train for one epoch
                train_loss, train_acc = self.train_epoch(train_loader, epoch)

                # Validate and adapt architecture
                valid_loss, valid_acc = self.validate(valid_loader)

                # Update learning rate
                self.scheduler.step()

                # Save best model
                if valid_acc > self.best_acc:
                    self.best_acc = valid_acc
                    self.save_checkpoint({
                        'epoch': epoch,
                        'state_dict': self.model.state_dict(),
                        'best_acc': self.best_acc,
                        'optimizer': self.optimizer.state_dict(),
                    }, is_best=True)

                print(f"\nEpoch Summary:")
                print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
                print(f"Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_acc:.2f}%")
                print(f"Best Acc: {self.best_acc:.2f}%")

        except Exception as e:
            print(f"Error during training: {str(e)}")
            import traceback
            traceback.print_exc()
            raise

    # Removed _generate_arch_weights as it's now handled by the model
    # def _generate_arch_weights(self):
    #     """Generate architecture weights for mixed operations"""
    #     weights_list = []
    #     for cell in self.model.search_space.cells:
    #         num_ops = len(cell._ops)
    #         # Use softmax of the learned architecture parameters
    #         weights = F.softmax(torch.stack([op.arch_parameters for op in cell._ops]), dim=-1)
    #         weights = weights.mean(dim=0) # Average weights across ops in a cell
    #         weights_list.append(weights)
    #     return weights_list


    def _accuracy(self, output, target, topk=(1,)):
        """Compute accuracy for top-k predictions"""
        maxk = max(topk)
        batch_size = target.size(0)

        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res

    def save_checkpoint(self, state, is_best):
        """Save checkpoint"""
        try:
            os.makedirs('checkpoints', exist_ok=True)
            filename = f"checkpoints/checkpoint_epoch{state['epoch']}.pth.tar"
            torch.save(state, filename)
            if is_best:
                best_filename = 'checkpoints/model_best.pth.tar'
                torch.save(state, best_filename)
                print(f"Saved best model checkpoint to {best_filename}")
        except Exception as e:
            print(f"Warning: Failed to save checkpoint: {str(e)}")

Overwriting ssdl/src/training/trainer.py


**Reasoning**:
The adaptation logic and the trainer have been updated to incorporate different adaptation strategies. Now, run the updated main script to test the training process with the new adaptation mechanisms.



In [17]:
%run ssdl/main.py

Script starting...
Starting main function...
Parsing arguments...
Loading configuration...
Setting random seed...
Getting device...
Using device: cpu

Starting MixedOp test...
Channels: in=3, out=8
Creating MixedOp...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights...
Number of operations: 4
Weights shape: torch.Size([4])
Performing forward pass...
Output shape: torch.Size([2, 8, 32, 32])
MixedOp test completed successfully!

Starting SearchSpace test...
Creating SearchSpace model...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights for cells...
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Performing forward pass...
Logits shape: torch.Size([2, 10])
Confidence shape: torch.Size([2, 1])

Testing original classifier...
Original classifier output shape: torch.Size([2, 10])
SearchSpace test completed successfully!

Starting Adapt

KeyboardInterrupt: 

In [ ]:
%run ssdl/main.py

Script starting...
Starting main function...
Parsing arguments...
Loading configuration...
Setting random seed...
Getting device...
Using device: cpu

Starting MixedOp test...
Channels: in=3, out=8
Creating MixedOp...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights...
Number of operations: 4
Weights shape: torch.Size([4])
Performing forward pass...
Output shape: torch.Size([2, 8, 32, 32])
MixedOp test completed successfully!

Starting SearchSpace test...
Creating SearchSpace model...
Creating dummy input...
Input shape: torch.Size([2, 3, 32, 32])
Creating weights for cells...
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Cell weights shape: torch.Size([2])
Performing forward pass...
Logits shape: torch.Size([2, 10])
Confidence shape: torch.Size([2, 1])

Testing original classifier...
Original classifier output shape: torch.Size([2, 10])
SearchSpace test completed successfully!

Starting Adapt